# Random Forest with Preprocessing

**What we will do (simple steps):**
1. Load data
2. Look at the data
3. Split into train and test
4. Clean the data (preprocessing)
5. Train a Random Forest
6. Check how good it is
7. Improve it
8. See which columns matter most

> Tip: To use your own data, change only **Step 1**.

## Step 0: Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

## Step 1: Load data

Here we make a small fake "customer churn" dataset (it has missing values and text columns, so we can practice cleaning).

**To use your own file**, remove the fake data code and use:
```python
df = pd.read_csv("your_file.csv")
```
Then set `target` to the name of the column you want to predict.

In [ ]:
np.random.seed(42)
n = 1000

df = pd.DataFrame({
    "age": np.random.randint(18, 70, n).astype(float),
    "monthly_income": np.random.normal(50000, 15000, n),
    "tenure_months": np.random.randint(1, 72, n).astype(float),
    "num_complaints": np.random.poisson(1.5, n).astype(float),
    "gender": np.random.choice(["Male", "Female"], n),
    "city": np.random.choice(["Delhi", "Mumbai", "Chennai", "Pune"], n),
    "plan": np.random.choice(["Basic", "Standard", "Premium"], n, p=[0.5, 0.3, 0.2]),
})

# Make the target (churn = 1 means customer left)
score = (
    0.6 * df["num_complaints"]
    - 0.04 * df["tenure_months"]
    + (df["plan"] == "Basic") * 0.8
    + np.random.normal(0, 1, n)
)
df["churn"] = (score > score.median()).astype(int)

# Add some missing values on purpose
for col in ["age", "monthly_income", "city"]:
    df.loc[df.sample(frac=0.08, random_state=1).index, col] = np.nan

target = "churn"   # <-- the column we want to predict
df.head()

## Step 2: Look at the data

In [ ]:
print("Shape (rows, columns):", df.shape)
print()
df.info()

In [ ]:
# How many missing values in each column?
df.isnull().sum()

In [ ]:
# How many 0 and 1 in the target?
df[target].value_counts()

## Step 3: Split into train and test

- **Train data** = the model learns from this (80%)
- **Test data** = we keep it hidden to check the model later (20%)

`stratify=y` keeps the same 0/1 ratio in both parts.

In [ ]:
X = df.drop(columns=[target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

## Step 4: Preprocessing (cleaning)

We treat number columns and text columns in different ways.

| Column type | What we do |
|---|---|
| Numbers | Fill missing with the **median**, then **scale** |
| Text (categories) | Fill missing with the **most common value**, then **one-hot encode** (turn text into 0/1 columns) |

Note: Random Forest does not really need scaling. We keep it here so you learn the full pattern (it is needed for models like Logistic Regression or KNN).

In [ ]:
numeric_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

print("Number columns:", numeric_cols)
print("Text columns:", categorical_cols)

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols),
])

## Step 5: Build the full pipeline and train

A **Pipeline** joins preprocessing + model in one object.
This is good because the same cleaning is applied to train and test data, and it avoids **data leakage**.

In [ ]:
model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("rf", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
])

model.fit(X_train, y_train)
print("Training done!")

## Step 6: Check the results

In [ ]:
y_pred = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print()
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["Stayed (0)", "Churned (1)"]).plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.show()

### Cross-validation
One split can be lucky or unlucky. Cross-validation tests the model on 5 different splits.

In [ ]:
cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring="accuracy")
print("Scores for each fold:", cv_scores.round(3))
print("Average:", cv_scores.mean().round(3))

## Step 7: Improve the model (hyperparameter tuning)

We try different settings and keep the best one.
- `n_estimators` = number of trees
- `max_depth` = how deep each tree can grow
- `min_samples_leaf` = minimum rows in a leaf (bigger = simpler model)

In [ ]:
param_grid = {
    "rf__n_estimators": [100, 200],
    "rf__max_depth": [None, 5, 10],
    "rf__min_samples_leaf": [1, 5],
}

grid = GridSearchCV(model, param_grid, cv=3, scoring="accuracy", n_jobs=-1)
grid.fit(X_train, y_train)

print("Best settings:", grid.best_params_)
print("Best CV score:", round(grid.best_score_, 3))

In [ ]:
best_model = grid.best_estimator_
y_pred_best = best_model.predict(X_test)

print("Test accuracy (best model):", round(accuracy_score(y_test, y_pred_best), 3))

## Step 8: Which columns matter most? (feature importance)

A higher number means the column helped the model more.

In [ ]:
# Get column names after preprocessing (one-hot makes new columns)
feature_names = best_model.named_steps["preprocess"].get_feature_names_out()
importances = best_model.named_steps["rf"].feature_importances_

fi = (
    pd.Series(importances, index=feature_names)
    .sort_values(ascending=True)
    .tail(10)
)

fi.plot(kind="barh", figsize=(7, 5), color="steelblue")
plt.title("Top 10 Important Features")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## Step 9: Save the model and predict new data

In [ ]:
import joblib

joblib.dump(best_model, "random_forest_model.pkl")
print("Model saved as random_forest_model.pkl")

In [ ]:
# Example: predict for a new customer (missing values are OK, the pipeline handles them)
new_customer = pd.DataFrame([{
    "age": 35,
    "monthly_income": 42000,
    "tenure_months": 6,
    "num_complaints": 4,
    "gender": "Male",
    "city": "Delhi",
    "plan": "Basic",
}])

loaded_model = joblib.load("random_forest_model.pkl")
pred = loaded_model.predict(new_customer)[0]
prob = loaded_model.predict_proba(new_customer)[0][1]

print("Prediction:", "Will churn" if pred == 1 else "Will stay")
print("Churn probability:", round(prob, 2))

## Summary

1. Split data first, then clean (to avoid leakage)
2. Use `Pipeline` + `ColumnTransformer` for clean, repeatable preprocessing
3. Check with cross-validation, not only one test split
4. Tune settings with `GridSearchCV`
5. Use feature importance to explain the model